# 03 可选 TARDIS 建模

这个 notebook 不再依赖 `notebooks/legacy/` 或遗留 `.dat` 数据。它从本地 `data/SN*/` 一维 FITS 光谱读取观测谱，并优先使用 `02_spectral_analysis_pipeline.ipynb` 写出的目标化分析表来给 TARDIS 生成第一版参数。

TARDIS 在本项目里只用于辅助谱线识别和定性比较谱形，不作为抛射物质量、丰度或爆炸能量的强约束。默认 `RUN_TARDIS=False`，先检查配置；确认参数后再在 `tardis` 环境中运行模拟。

In [ ]:
%matplotlib inline

from pathlib import Path
import shutil
import sys
from importlib import reload

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src import spectral_notebook_tools as snt

ANALYSIS_DIR = PROJECT_ROOT / "output" / "analysis_pipeline"

## 1. 目标和手动覆盖参数

先只改这个 cell。`ANALYSIS_TAG` 为空时会自动读取该目标最新的 `*_target_status.csv`、`*_line_diagnostics_qc.csv` 等分析产物；如果你想强制使用某次输出，可以填 `SN2026KID` 或你在 02 里设置的 `OUTPUT_TAG`。

手动覆盖项优先级最高。`MANUAL_APPARENT_MAG` 用于由红移和视星等粗估 luminosity；如果你已经知道更合适的 bolometric luminosity，直接填 `MANUAL_LOG_LSUN`。

In [ ]:
TARGET = "SN2026KID"
ANALYSIS_TAG = ""  # 空字符串表示按目标读取最新产物；也可填 02 的 OUTPUT_TAG/RUN_TAG
SPECTRUM_INDEX = 0

RUN_TARDIS = False  # 确认配置和原子数据后再改 True

MANUAL_Z = np.nan
MANUAL_TYPE = ""
MANUAL_VELOCITY_KMS = np.nan
MANUAL_EPOCH_DAYS = np.nan
MANUAL_APPARENT_MAG = np.nan
MANUAL_LOG_LSUN = np.nan

BASE_CONFIG_PATH = PROJECT_ROOT / "configs" / "tardis" / "base_Ia.yml"

## 2. 从 02 的科学产物估计 TARDIS 起点

本步读取的是当前项目产物，不读取 `legacy/`。优先使用手动红移表，其次目标状态表和光谱摘要表；类型来自目标状态/光谱摘要；速度来自 `line_diagnostics_qc.csv` 中 `adopt/check` 的关键谱线；epoch 和 luminosity 是粗略起点，通常需要人工调整。

In [ ]:
reload(snt)

context = snt.estimate_tardis_context(
    PROJECT_ROOT,
    TARGET,
    analysis_tag=ANALYSIS_TAG or None,
    spectrum_index=SPECTRUM_INDEX,
    manual_z=MANUAL_Z,
    manual_type=MANUAL_TYPE,
    manual_velocity_kms=MANUAL_VELOCITY_KMS,
    manual_epoch_days=MANUAL_EPOCH_DAYS,
    manual_apparent_mag=MANUAL_APPARENT_MAG,
    manual_log_lsun=MANUAL_LOG_LSUN,
)

display(snt.spectrum_choice_table(context["spectra"], TARGET))
display(snt.tardis_context_table(context))

for name, table in context["analysis_tables"].items():
    if not table.empty:
        print(f"{name}: rows={len(table)}, product_file={table.get('product_file', pd.Series([''])).iloc[0]}")

## 3. 检查选中的观测光谱

下轴是观测波长，上轴是按当前采用红移换算后的静止系波长。这个图只用于检查 TARDIS 要比较哪一条真实观测光谱。

In [ ]:
spec = context["spectrum"]
z = context["z"]

fig, ax = plt.subplots(figsize=(10.5, 4.6))
ax.plot(spec["wave"], snt.normalize_for_comparison(spec["flux"]), color="black", lw=0.8)
ax.set_xlabel("Observed wavelength (Angstrom)")
ax.set_ylabel("Normalized flux")
ax.set_title(f"{context['target']} observed spectrum for TARDIS setup")
ax.grid(alpha=0.25)
snt.add_rest_top_axis(ax, z)
snt.show_figure(fig)

print(f"selected file = {spec['file']}")
print(f"z = {z:.6f} ({context['z_source']})")

## 4. 生成 TARDIS YAML 配置

配置从 `configs/tardis/base_Ia.yml` 复制并覆盖：`luminosity_requested`、`time_explosion`、速度范围和 `atom_data` 绝对路径。非 Ia 目标会使用一个非常粗略的 II/Ibc 均匀丰度 preset；这只是定性起点，不代表自动物理拟合。

In [ ]:
CONFIG_PATH = PROJECT_ROOT / "configs" / "tardis" / f"{context['target']}.yml"
config, config_path = snt.build_tardis_config_from_context(
    context,
    project_root=PROJECT_ROOT,
    base_config_path=BASE_CONFIG_PATH,
    output_config_path=CONFIG_PATH,
)

print(f"wrote config: {config_path}")
display(pd.DataFrame([
    {"section": "supernova", **config["supernova"]},
    {"section": "velocity", **config["model"]["structure"]["velocity"]},
    {"section": "abundances", **config["model"]["abundances"]},
]))

## 5. 检查 TARDIS 原子数据

TARDIS 需要 `data/kurucz_cd23_chianti_H_He_latest.h5`。如果缺失，在终端运行：

```bash
conda activate tardis
python scripts/download_tardis_atom_data.py
```

In [ ]:
ATOM_DATA_FILE = Path(config["atom_data"])
print(f"atom_data = {ATOM_DATA_FILE}")

if ATOM_DATA_FILE.exists():
    print(f"found atomic data: {ATOM_DATA_FILE.stat().st_size / 1e6:.0f} MB")
else:
    print("atomic data missing; run scripts/download_tardis_atom_data.py in the tardis environment")

## 6. 可选：运行 TARDIS

只有 `RUN_TARDIS=True` 时才会导入并运行 TARDIS。运行时间取决于 packet 数和迭代数；如果只是检查 notebook 流程，保持 False。

In [ ]:
tardis_wave = None
tardis_flux = None
sim = None

if RUN_TARDIS:
    if not ATOM_DATA_FILE.exists():
        raise RuntimeError(f"Atomic data file not found: {ATOM_DATA_FILE}")
    from tardis import run_tardis

    print(f"running TARDIS with {config_path}")
    try:
        sim = run_tardis(str(config_path), show_convergence_plots=False, log_level="WARNING", show_progress_bars=False)
    except TypeError:
        sim = run_tardis(str(config_path), show_convergence_plots=False, log_level="WARNING")
    tardis_wave, tardis_flux = snt.extract_tardis_spectrum_arrays(sim)
    print("simulation complete")
    print(f"iterations_executed = {getattr(sim, 'iterations_executed', 'unknown')}")
    print(f"spectrum points = {len(tardis_wave)}")
else:
    print("RUN_TARDIS=False：已跳过模拟，只生成/检查配置。")

## 7. 保存模拟结果并比较谱形

如果刚刚运行了 TARDIS，本步会保存当前模拟光谱和配置副本到 `output/<target>/tardis/`。如果没有运行，但该目录已经有当前目标的 TARDIS 输出，本步可以读取并显示已有结果；这只是当前项目输出，不依赖 `legacy/`。

In [ ]:
TARDIS_DIR = PROJECT_ROOT / "output" / context["target"] / "tardis"
TARDIS_DIR.mkdir(parents=True, exist_ok=True)
SPECTRUM_OUT = TARDIS_DIR / f"tardis_spectrum_{context['target']}.dat"
CONFIG_COPY = TARDIS_DIR / f"tardis_config_{context['target']}.yml"
COMPARISON_OUT = TARDIS_DIR / f"tardis_comparison_{context['target']}.png"

if tardis_wave is not None and tardis_flux is not None:
    np.savetxt(SPECTRUM_OUT, np.column_stack([tardis_wave, tardis_flux]), header="wavelength_A luminosity_density_lambda_erg_s_A")
    shutil.copyfile(config_path, CONFIG_COPY)
    print(f"saved {SPECTRUM_OUT}")
    print(f"saved {CONFIG_COPY}")
elif SPECTRUM_OUT.exists():
    existing = np.loadtxt(SPECTRUM_OUT)
    tardis_wave, tardis_flux = existing[:, 0], existing[:, 1]
    print(f"loaded existing current-project output: {SPECTRUM_OUT}")
else:
    print("还没有 TARDIS 光谱。把 RUN_TARDIS 改成 True 后重新运行第 6-7 节。")

if tardis_wave is not None and tardis_flux is not None:
    comparison_fig = snt.plot_tardis_comparison(
        context["spectrum"],
        tardis_wave,
        tardis_flux,
        z=context["z"],
        target=context["target"],
        output_path=COMPARISON_OUT,
    )
    snt.show_figure(comparison_fig)

## 8. 调参边界

优先调 `MANUAL_Z`、`MANUAL_TYPE`、`MANUAL_VELOCITY_KMS`、`MANUAL_EPOCH_DAYS` 和 luminosity。若只是几条谱线对不上，不要直接把 TARDIS 结果解释成物理参数；本项目的稀疏光谱更适合把 TARDIS 当作谱线识别和连续谱形状的 sanity check。